## Preprocessing of RetailRocket: from event logs to session sequences

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('../../data/events.csv')
print(df.shape)
df.head()

(2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [6]:
# CLEAN DATA

# Convert Unix timestamps (milliseconds) into datetime
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    unit="ms",
    errors="coerce",
    utc=True
)

# Standardize event names
df["event"] = df["event"].astype("string").str.strip().str.lower()

# Remove rows without essential information
df = df.dropna(
    subset=["visitorid", "timestamp", "event"]
)

# Remove exact duplicate records
before_duplicates = len(df)
df = df.drop_duplicates()
duplicates_removed = before_duplicates - len(df)

# Sort chronologically within each visitor
df = df.sort_values(
    ["visitorid", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Duplicates removed:", duplicates_removed)
print("\nEvent distribution:")
print(df["event"].value_counts())


# RECONSTRUCT SESSIONS !! Needed in this dataset

GAP_MIN = 60

# Calculate inactivity gap between consecutive events
time_diff = df.groupby("visitorid")["timestamp"].diff()

# Start a new session if:
# - This is the visitor's first event
# - The inactivity gap exceeds 30 minutes

new_session = (
    df["visitorid"].ne(df["visitorid"].shift()) |
    time_diff.gt(pd.Timedelta(minutes=GAP_MIN))
)

# Generate globally unique session IDs
df["session"] = new_session.cumsum().astype("int64")

print("\nNumber of reconstructed sessions:",
      df["session"].nunique())



# BUILD SESSION SEQUENCES

# Identify whether each event represents a purchase
df["is_purchase"] = df["event"].eq("transaction")

sessions = (
    df.groupby("session", sort=False)
      .agg(
          user=("visitorid", "first"),
          events=("event", list),
          has_purchase=("is_purchase", "any")
      )
      .reset_index()
)

# Number of events per session
sessions["n_events"] = sessions["events"].str.len()

# Session outcome
sessions["outcome"] = np.where(
    sessions["has_purchase"],
    "purchase",
    "no_purchase"
)

# RetailRocket does not provide monetary transaction amounts.
# Zero means no recorded purchase; NaN means purchase amount unknown.
sessions["total_amount"] = np.where(
    sessions["has_purchase"],
    np.nan,
    0.0
)

# Remove temporary column
sessions = sessions.drop(columns=["has_purchase"])

# Arrange columns to match the previous dataset
sessions = sessions[
    [
        "session",
        "user",
        "events",
        "outcome",
        "total_amount",
        "n_events"
    ]
]


# FILTER SESSIONS

# Keep sessions with at least two events.
# Do not automatically remove long sessions using a percentile:
# investigate them first.

MIN_EVENTS = 2

sessions = sessions[
    sessions["n_events"] >= MIN_EVENTS
].copy()

sessions = sessions.reset_index(drop=True)


# VALIDATION

print("\n========== SESSION SUMMARY ==========")

print("Total sessions:", len(sessions))
print("Unique users:", sessions["user"].nunique())

print("\nSession length statistics:")
print(sessions["n_events"].describe())

print("\nSession outcome distribution:")
print(sessions["outcome"].value_counts())

print("\nSession length distribution:")
print(sessions["n_events"].value_counts().sort_index())

print("\nFinal dataset:")
display(sessions.head())


# PREPARE SEQUENCES FOR LEVENSHTEIN

sequences = sessions["events"].tolist()

print("\nNumber of sequences:", len(sequences))
print("Example sequence:", sequences[0])


Cleaned shape: (2755641, 7)
Duplicates removed: 0

Event distribution:
event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: Int64

Number of reconstructed sessions: 1726714

========== SESSION SUMMARY ==========
Total sessions: 386033
Unique users: 324236

Session length statistics:
count    386033.000000
mean          3.665386
std           6.247261
min           2.000000
25%           2.000000
50%           2.000000
75%           4.000000
max         422.000000
Name: n_events, dtype: float64

Session outcome distribution:
outcome
no_purchase    372919
purchase        13114
Name: count, dtype: int64

Session length distribution:
n_events
2      211815
3       75228
4       34541
5       19548
6       12021
        ...  
305         1
310         1
321         1
330         1
422         1
Name: count, Length: 232, dtype: int64

Final dataset:


,session,user,events,outcome,total_amount,n_events
0,1,0,"[view, view, view]",no_purchase,0.0,3
1,3,2,"[view, view, view, view, view, view, view, view]",no_purchase,0.0,8
2,7,6,"[addtocart, view, view, view, view]",no_purchase,0.0,5
3,9,7,"[view, view]",no_purchase,0.0,2
4,16,13,"[view, view]",no_purchase,0.0,2



Number of sequences: 386033
Example sequence: ['view', 'view', 'view']


### Validation 

In [5]:

# Session duration
session_stats = (
    df.groupby("session")
      .agg(
          start=("timestamp", "min"),
          end=("timestamp", "max"),
          n_events=("event", "size")
      )
)

session_stats["duration_min"] = (
    session_stats["end"] - session_stats["start"]
).dt.total_seconds() / 60

print("Median session duration:",
      session_stats["duration_min"].median(), "minutes")

print("90th percentile duration:",
      session_stats["duration_min"].quantile(0.90), "minutes")

print("Maximum session duration:",
      session_stats["duration_min"].max(), "minutes")

print("\nSessions longer than 2 hours:",
      (session_stats["duration_min"] > 120).mean() * 100, "%")

# Check whether sessions respect the inactivity threshold
gaps = df.groupby("session")["timestamp"].diff()

print("\nGaps exceeding 30 minutes:",
      (gaps > pd.Timedelta(minutes=30)).sum())


Median session duration: 0.0 minutes
90th percentile duration: 3.5527066666666727 minutes
Maximum session duration: 728.4041333333333 minutes

Sessions longer than 2 hours: 0.058637376360565936 %

Gaps exceeding 30 minutes: 0


In [8]:
# Analyse session duration and number of events together
session_check = (
    df.groupby("session")
      .agg(
          user=("visitorid", "first"),
          start=("timestamp", "min"),
          end=("timestamp", "max"),
          n_events=("event", "size"),
          unique_timestamps=("timestamp", "nunique")
      )
)

session_check["duration_min"] = (
    session_check["end"] - session_check["start"]
).dt.total_seconds() / 60

# Overall statistics for sessions with at least 2 events
multi_event = session_check[session_check["n_events"] >= 2]

print("Total sessions:", len(session_check))
print("Single-event sessions:", (session_check["n_events"] == 1).sum())
print("Multi-event sessions:", len(multi_event))

print("\nDuration statistics (sessions with >= 2 events):")
print(multi_event["duration_min"].describe())

print("\nMulti-event sessions with zero duration:",
      (multi_event["duration_min"] == 0).sum())


Total sessions: 1726714
Single-event sessions: 1340681
Multi-event sessions: 386033

Duration statistics (sessions with >= 2 events):
count    386033.000000
mean         11.856401
std          28.006078
min           0.000000
25%           1.013817
50%           3.369800
75%          12.132550
max         776.733383
Name: duration_min, dtype: float64

Multi-event sessions with zero duration: 159


In [9]:

# Inspect the 10 longest sessions
long_sessions = session_check.sort_values(
    "duration_min", ascending=False
).head(10)

print(long_sessions)


            user                            start  \
session                                             
636881    519839 2015-07-07 14:12:04.253000+00:00   
636882    519839 2015-07-08 14:31:37.090000+00:00   
303485    247235 2015-08-18 14:39:47.579000+00:00   
430041    350566 2015-06-05 14:49:02.910000+00:00   
455748    371606 2015-07-14 14:48:39.917000+00:00   
1590200  1297062 2015-07-08 14:49:50.493000+00:00   
1688859  1376763 2015-07-20 14:51:16.779000+00:00   
1359189  1108631 2015-08-11 14:43:01.385000+00:00   
938235    765630 2015-06-21 14:53:47.836000+00:00   
791443    645525 2015-08-20 14:47:30.407000+00:00   

                                     end  n_events  unique_timestamps  \
session                                                                 
636881  2015-07-08 03:08:48.256000+00:00       321                318   
636882  2015-07-09 02:50:01.765000+00:00       221                219   
303485  2015-08-19 02:55:08.460000+00:00       143                141  

In [10]:
# Sessions containing only transaction and/or addtocart events
non_view_sessions = sessions[
    sessions["events"].apply(
        lambda events: len(events) >= 2
        and all(e in ["transaction", "addtocart"] for e in events)
    )
]

print("Number of sessions:", len(non_view_sessions))

display(
    non_view_sessions[
        ["session", "user", "events", "outcome", "n_events"]
    ].head(30)
)

Number of sessions: 1367


,session,user,events,outcome,n_events
43,204,155,"[addtocart, addtocart, addtocart, addtocart, a...",no_purchase,6
367,1693,1399,"[addtocart, addtocart]",no_purchase,2
838,3731,2985,"[addtocart, addtocart, addtocart, addtocart]",no_purchase,4
851,3810,3048,"[addtocart, transaction]",purchase,2
870,3886,3104,"[addtocart, addtocart, transaction, transaction]",purchase,4
1335,6110,4960,"[addtocart, addtocart, addtocart]",no_purchase,3
1424,6537,5321,"[addtocart, addtocart]",no_purchase,2
1527,7004,5703,"[transaction, transaction]",purchase,2
1706,7835,6390,"[addtocart, transaction]",purchase,2
1879,8633,7016,"[transaction, transaction]",purchase,2


In [11]:
# Only transaction events
transaction_only = non_view_sessions[
    non_view_sessions["events"].apply(
        lambda events: all(e == "transaction" for e in events)
    )
]

# Only add-to-cart events
addtocart_only = non_view_sessions[
    non_view_sessions["events"].apply(
        lambda events: all(e == "addtocart" for e in events)
    )
]

# Mixtures of transaction and add-to-cart
mixed_sessions = non_view_sessions[
    non_view_sessions["events"].apply(
        lambda events: "transaction" in events and "addtocart" in events
    )
]

print("Transaction only:", len(transaction_only))
print("Add-to-cart only:", len(addtocart_only))
print("Mixed:", len(mixed_sessions))

Transaction only: 96
Add-to-cart only: 599
Mixed: 672


- Transaction only: the visitor may have viewed products before the reconstructed session began, or the session may contain incomplete browsing data.
- Add-to-cart only: the visitor may have added products without generating a recorded view during that session.
- Mixed: the visitor added products and completed transactions, but no product views were recorded.

Checking proportions..

In [12]:
print("Total sessions:", len(sessions))

print(
    "Sessions without views:",
    len(non_view_sessions),
    f"({len(non_view_sessions) / len(sessions) * 100:.2f}%)"
)

print(
    "Transaction-only:",
    f"{len(transaction_only) / len(sessions) * 100:.2f}%"
)

print(
    "Add-to-cart-only:",
    f"{len(addtocart_only) / len(sessions) * 100:.2f}%"
)

print(
    "Mixed:",
    f"{len(mixed_sessions) / len(sessions) * 100:.2f}%"
)

Total sessions: 386033
Sessions without views: 1367 (0.35%)
Transaction-only: 0.02%
Add-to-cart-only: 0.16%
Mixed: 0.17%


In [14]:
display(
    df[df["session"].isin(transaction_only["session"])]
    .sort_values(["session", "timestamp"])
    [["session", "visitorid", "timestamp", "event", "itemid", "transactionid"]]
    .head(100)
)

,session,visitorid,timestamp,event,itemid,transactionid
10874,7004,5703,2015-07-27 21:03:15.619000+00:00,transaction,186214,3297.0
10875,7004,5703,2015-07-27 21:03:15.619000+00:00,transaction,295495,3297.0
13672,8633,7016,2015-06-17 00:16:12.144000+00:00,transaction,411039,14920.0
13673,8633,7016,2015-06-17 00:16:12.160000+00:00,transaction,32591,14920.0
23322,14969,12148,2015-06-02 15:48:03.007000+00:00,transaction,157438,5921.0
...,...,...,...,...,...,...
975921,609946,497827,2015-07-29 02:02:40.643000+00:00,transaction,379953,12084.0
975922,609946,497827,2015-07-29 02:02:41.127000+00:00,transaction,3383,12084.0
976145,610089,497935,2015-05-12 20:06:08.793000+00:00,transaction,4887,2100.0
976146,610089,497935,2015-05-12 20:06:08.856000+00:00,transaction,416187,2100.0


In [15]:
transaction_records = df[
    df["session"].isin(transaction_only["session"])
]

print("Missing transaction IDs:",
      transaction_records["transactionid"].isna().sum())

print("Unique transaction IDs:",
      transaction_records["transactionid"].nunique())

Missing transaction IDs: 0
Unique transaction IDs: 101


The important observations:
- Only 0.35% of sessions lack a product-view event.
- Transaction-only sessions represent just 0.02% of all sessions.
- Add-to-cart-only and mixed sessions are also very small proportions.
- All transaction events have a non-missing transaction ID.
This suggests that sessions without views are an unusual but relatively small part of the reconstructed dataset.

In [16]:
# Inspect transaction IDs appearing in multiple sessions
transaction_sessions = (
    df[df["event"] == "transaction"]
    .groupby("transactionid")
    .agg(
        n_sessions=("session", "nunique"),
        n_events=("event", "size"),
        n_visitors=("visitorid", "nunique")
    )
    .sort_values("n_sessions", ascending=False)
)

display(transaction_sessions.head(20))

,n_sessions,n_events,n_visitors
transactionid,,,
5226.0,2,2,1
3011.0,2,3,1
12847.0,2,12,1
13733.0,2,2,1
14083.0,2,4,1
3.0,1,1,1
4.0,1,1,1
5.0,1,1,1
6.0,1,1,1


Finally importing dataset...

In [17]:
from pathlib import Path

# Output directory
output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# CSV export
sessions.to_csv(
    output_dir / "retailrocket_sessions.csv",
    index=False
)

# Parquet export (recommended)
sessions.to_parquet(
    output_dir / "retailrocket_sessions.parquet",
    index=False
)

print("Files exported successfully:")
print(output_dir / "retailrocket_sessions.csv")
print(output_dir / "retailrocket_sessions.parquet")


Files exported successfully:
..\..\data\processed\retailrocket_sessions.csv
..\..\data\processed\retailrocket_sessions.parquet
